# Odds ratio table post-processing

Here we:

- Harmonize the OR tables (make sure columns match)
- Patch in TP53 ExCALIBR ORs
- Keep only "Missense" vs "All" consequences for ExCALIBR, VEPs, and Combined points
- Add p-values adjusted for multiple testing within gene/dataset
- Reformat the count reporting
- Add variant counts
- Censor out all ORs where carrier or control counts <20

In [1]:
# Common imports
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

In [91]:
# Constants
index_cols = ['Dataset/Predictor', 'Gene', 'Grouping/Calibration', 'Consequence', 'Group/Range']

# Input OR files
functional_file = "results/functional-OR-estimates_2026-10-07.parquet"
excalibr_file = "results/excalibr-OR-estimates_2026-10-02.parquet"
excalibr_TP53_file = "results/excalibr-TP53-OR-estimates_2026-10-05.parquet"
vep_file = "results/VEP-OR-estimates_2026-10-02.parquet"
tp_file = "results/total-points-OR-estimates_2025-10-02.parquet"

# Input variant count files
functional_vc_file = "variant_groups/functional-variant-counts_2026-10-10.parquet"
excalibr_vc_file = "variant_groups/excalibr-variant-counts_2026-10-10.parquet"
vep_vc_file = "variant_groups/vep-variant-counts_2026-10-10.parquet"
tp_vc_file = "variant_groups/tp-variant-counts_2026-10-10.parquet"

# Output harmonized files
harmonized_or_file = "results/all-OR-estimates_2026-10-12.parquet"
deliverable_or_file = "results/AoU-OR-estimates_2026-10-12.tsv.gz"
deliverable_xlsx = "results/Supplementary-Data-2.xlsx"

## Harmonization
Harmonize column names across dataframes.
The harmonized index column names are:

In [3]:
index_cols

### ExCALIBR
- name the index columns

In [4]:
excalibr_df = pd.read_parquet(excalibr_file)
excalibr_df

In [5]:
excalibr_df.index.set_names(
    index_cols,
    inplace=True
)
excalibr_df

Drop exrtaneous consequences

In [6]:
# Check Consequence values
excalibr_df.index.to_frame()['Consequence'].value_counts()

In [7]:
excalibr_df.drop(index='Stop Gained', level='Consequence', inplace=True)
# Re-check Consequence values
excalibr_df.index.to_frame()['Consequence'].value_counts()

#### Patch in TP53 file

In [8]:
excalibr_TP53_df = pd.read_parquet(excalibr_TP53_file)
excalibr_TP53_df

In [9]:
# Check Consequence values
excalibr_TP53_df.index.to_frame()['Consequence'].value_counts()

In [10]:
excalibr_TP53_df.drop(index='Stop Gained', level='Consequence', inplace=True)
# Re-check Consequence values
excalibr_TP53_df.index.to_frame()['Consequence'].value_counts()

In [11]:
# Check for index overlap
excalibr_TP53_df.index.isin(excalibr_df.index).any()

In [12]:
# Merge
excalibr_df = pd.concat([excalibr_df, excalibr_TP53_df])

In [13]:
# Re-check for index overlap
excalibr_TP53_df.index.isin(excalibr_df.index).any()

In [14]:
# Check gene coverage
excalibr_df.index.to_frame()['Gene'].value_counts()

### VEP
- rename the index columns

In [15]:
vep_df = pd.read_parquet(vep_file)
vep_df

In [16]:
vep_df.index.set_names(
    index_cols,
    inplace=True
)
vep_df

In [17]:
# Check Consequence values
vep_df.index.to_frame()['Consequence'].value_counts()

In [18]:
# Check gene coverage
vep_df.index.to_frame()['Gene'].value_counts()

### Functional
- already has correct column names

In [19]:
functional_df = pd.read_parquet(functional_file)
functional_df

In [20]:
functional_df.index.to_frame()['Gene'].value_counts()

n.b. LDLR data does not have functional calls.

### Combined points
- already has correct column names

In [21]:
tp_df = pd.read_parquet(tp_file)
tp_df

In [22]:
# Check Consequence values
tp_df.index.to_frame()['Consequence'].value_counts()

In [23]:
tp_df.drop(index='Stop Gained', level='Consequence', inplace=True)
# Re-check Consequence values
tp_df.index.to_frame()['Consequence'].value_counts()

## Multiple testing correction

We apply Holm-Bonferroni to each dataset/gene and use hierarchical testing for nested groups.

In [24]:
def add_holm_adjusted_p(df, p_value_col='p-value', alpha=0.005):
    df = df.copy()
    df['Significant (unadjusted)'] = df[p_value_col] <= alpha
    df['Significant (Holm adjusted)'], df['p-value (Holm adjusted)'], _, _ = multipletests(
        df[p_value_col], alpha=alpha, method='holm'
    )
    return df

### ExCALIBR
Grouped by dataset and consequence

In [25]:
excalibr_df.index.to_frame().Consequence.value_counts()

In [26]:
excalibr_df_adjusted = excalibr_df.groupby(
    ['Dataset/Predictor', 'Consequence'],
    group_keys=False,
).apply(
    add_holm_adjusted_p,
    include_groups=False
)
excalibr_df_adjusted

### Functional
Grouped by dataset only

In [27]:
functional_df_adjusted = functional_df.groupby(
    'Dataset/Predictor',
    group_keys=False,
).apply(add_holm_adjusted_p)
functional_df_adjusted

### VEP
Grouped by predictor, calibration method, and consequence

In [28]:
vep_df.index.to_frame().Consequence.value_counts()

In [29]:
vep_df_adjusted = vep_df.groupby(
    ['Dataset/Predictor', 'Grouping/Calibration', 'Consequence'],
    group_keys=False,
).apply(add_holm_adjusted_p)
vep_df_adjusted

### Combined points

Grouped by gene and consequence

In [30]:
tp_df.index.to_frame().Consequence.value_counts()

In [31]:
tp_df_adjusted = tp_df.groupby(
    ['Gene', 'Consequence'],
    group_keys=False,
).apply(
    add_holm_adjusted_p,
    include_groups=False
)
tp_df_adjusted

## Merge in variant counts

### Functional

In [32]:
functional_vc_df = pd.read_parquet(functional_vc_file)
functional_vc_df

In [33]:
# Set column names
functional_vc_df.index.set_names(index_cols, inplace=True)

In [34]:
# Fill zeros and force integers
functional_vc_df = functional_vc_df.fillna(0).apply(pd.to_numeric, downcast="integer")

In [35]:
# Check index match
functional_ors_covered_by_vc = functional_df_adjusted.index.isin(functional_vc_df.index)
if functional_ors_covered_by_vc.all():
    print("VC DF covers OR df (good!)")
else:
    display(functional_df.index.to_frame()[~functional_ors_covered_by_vc])

In [36]:
functional_with_counts_df = pd.concat([functional_df_adjusted, functional_vc_df], axis='columns')
functional_with_counts_df

### ExCALIBR

In [37]:
excalibr_vc_df = pd.read_parquet(excalibr_vc_file)
excalibr_vc_df

In [38]:
# Fix capitalization to "Valriants in group"
excalibr_vc_df.rename(columns={'Variants in Group': 'Variants in group'}, inplace=True)

In [39]:
# Fill zeros and force integers
excalibr_vc_df = excalibr_vc_df.fillna(0).apply(pd.to_numeric, downcast="integer")

In [40]:
# Check index match
excalibr_ors_covered_by_vc = excalibr_df_adjusted.index.isin(excalibr_vc_df.index)
if excalibr_ors_covered_by_vc.all():
    print("VC DF covers OR df (good!)")
else:
    display(excalibr_df.index.to_frame()[~excalibr_ors_covered_by_vc])

In [41]:
excalibr_with_counts_df = pd.concat([excalibr_df_adjusted, excalibr_vc_df], axis='columns')
excalibr_with_counts_df

### VEP

In [42]:
vep_vc_df = pd.read_parquet(vep_vc_file)
vep_vc_df

In [43]:
# Fix capitalization to "Valriants in group"
vep_vc_df.rename(columns={'Variants in Group': 'Variants in group'}, inplace=True)

In [44]:
# Fill zeros and force integers
vep_vc_df = vep_vc_df.fillna(0).apply(pd.to_numeric, downcast="integer")

In [45]:
# Check index match
vep_ors_covered_by_vc = vep_df_adjusted.index.isin(vep_vc_df.index)
if vep_ors_covered_by_vc.all():
    print("VC DF covers OR df (good!)")
else:
    display(vep_df.index.to_frame()[~vep_ors_covered_by_vc])

In [46]:
vep_with_counts_df = pd.concat([vep_df_adjusted, vep_vc_df], axis='columns')
vep_with_counts_df

### Combined Points

In [47]:
tp_vc_df = pd.read_parquet(tp_vc_file)
tp_vc_df

In [48]:
# Fix capitalization to "Valriants in group"
tp_vc_df.rename(columns={'Variants in Group': 'Variants in group'}, inplace=True)

In [49]:
# Fill zeros and force integers
tp_vc_df = tp_vc_df.fillna(0).apply(pd.to_numeric, downcast="integer")

In [50]:
# Check index match
tp_ors_covered_by_vc = tp_df_adjusted.index.isin(tp_vc_df.index)
if tp_ors_covered_by_vc.all():
    print("VC DF covers OR df (good!)")
else:
    display(tp_df.index.to_frame()[~tp_ors_covered_by_vc])

In [51]:
tp_with_counts_df = pd.concat([tp_df_adjusted, tp_vc_df], axis='columns')
tp_with_counts_df

## Reformat count reporting

Instead of reporting cases/controls with/without variants, report total cases+controls and carrier cases/controls

In [52]:
or_df = pd.concat((functional_with_counts_df, excalibr_with_counts_df, vep_with_counts_df, tp_with_counts_df))

In [53]:
or_df['Total cases'] = or_df['Cases with variants'] + or_df['Cases without variants']
or_df['Total controls'] = or_df['Controls with variants'] + or_df['Controls without variants']

In [54]:
leq20_carrier_cases = or_df['Cases with variants'] <= 20
leq20_control_cases = or_df['Controls with variants'] <= 20
or_df['Censor'] = leq20_carrier_cases | leq20_control_cases

In [55]:
or_df['Carrier cases'] = or_df['Cases with variants'].astype(str).mask(
    leq20_carrier_cases & (or_df['Cases with variants'] > 0),
    '≤ 20'
)

In [56]:
or_df['Carrier controls'] = or_df['Controls with variants'].astype(str).mask(
   leq20_control_cases & (or_df['Controls with variants'] > 0),
    '≤ 20'
)

## Update row and column labels

In [57]:
or_df.rename(
    index={
        'Combined Points': 'Combined points',
        'single_gene': 'Gene-specific',
        'genome_wide': 'Genome-wide aggregation',
        'domain_aggregate': 'Domain aggregation',
        'ExCALIBR + REVEL': 'ExCALIBR + REVEL single-gene',
        'Functionally Abnormal': 'Functionally abnormal',
        'Functionally Normal': 'Functionally normal',
        #'Stop Gained': 'Stop gained',
        #'Stop Gained / Deletion / Splicing': 'Stop gained / deletion / splicing'
    },
    columns={'p-value': 'Wald p-value'},
    inplace=True
)

In [58]:
# Exponentiate Logs
or_df[['Odds ratio', 'OR_LI', 'OR_UI']] = np.exp(or_df[['LogOR', 'LogOR_LI', 'LogOR_UI']])

In [59]:
or_df

In [60]:
(or_df[['Variants in group']] == 0).value_counts()

## Order rows nicely

In [82]:
new_index['Group/Range'].drop_duplicates().to_list()

In [85]:
new_index = or_df.index.to_frame()
new_index['Grouping/Calibration'] = pd.Categorical(
    new_index['Grouping/Calibration'],
    categories = [
        'Functional class',
        'ExCALIBR',
        'Domain aggregation',
        'Genome-wide aggregation',
        'Gene-specific',
        'ExCALIBR + REVEL single-gene'
    ],
    ordered=True
)
new_index['Group/Range'] = pd.Categorical(
    new_index['Group/Range'],
    categories = [
        'Functionally abnormal',
        'Functionally normal',
    ] + [
        f'≤ -{points}' for points in range(12,0,-1)
    ] + [
        f'≥ +{points}' for points in range(1,13,1)
    ],
    ordered=True
)

or_df = or_df.set_index(
    pd.MultiIndex.from_frame(new_index)
).sort_index(
    axis='index',
    level=['Grouping/Calibration', 'Dataset/Predictor', 'Gene', 'Consequence', 'Group/Range']
)


## Save full harmonized file

In [86]:
or_df.to_parquet(harmonized_or_file)

## Censor

In [88]:
deliverable_df = or_df.loc[
    or_df['Variants in group'] > 0,
    [
        'Odds ratio',
        'OR_LI',
        'OR_UI',
        'LogOR',
        'LogOR_LI',
        'LogOR_UI',
        'Wald p-value',
        'p-value (Holm adjusted)',
        'Cohort size',
        'Total cases',
        'Total controls',
        'Carrier cases',
        'Carrier controls',
        'Variants in group',
        'Case only variant count',
        'Control only variant count',
        'Overlap variant count',
        'Variants in cohort'
    ]
].copy()

deliverable_df.loc[
    or_df['Censor'],
    ['LogOR', 'LogOR_LI', 'LogOR_UI', 'Odds ratio', 'OR_LI', 'OR_UI']
] = None

deliverable_df

In [89]:
deliverable_df.to_csv(deliverable_or_file, sep='\t')

In [93]:
# Not installed by default
# !pip install openpyxl

In [94]:
deliverable_df.to_excel(deliverable_xlsx)